# News Topic Classification on AG News
## Sequential (BiLSTM) vs. Transformer Encoder — both implemented from scratch

**Case study — Transformer Modeling and Sequence Learning (Topic 2)**

**Research question.** *How do recurrent networks and Transformers compare at classifying short news
texts?*

**Dataset.** AG News (Kaggle: `amananandrai/ag-news-classification-dataset`) — 120,000 training and
7,600 test articles over four topics: World, Sports, Business, Sci/Tech. Following the brief, a
**controlled 60,000-record** stratified subset is used for training/validation, while the **full
official 7,600-record test set** is held back for the final comparison.

**What "from scratch" means here.** PyTorch is used for tensors, autograd and the optimiser.
`nn.Linear`, `nn.Embedding` and `nn.Dropout` are used purely as parameter containers. Every
sequence-modelling component is written out by hand:

| Component | Written by hand in this notebook |
|---|---|
| LSTM cell (input/forget/cell/output gates) | `LSTMCellScratch` |
| Recurrence over time + bidirectionality | `LSTMLayerScratch` |
| Scaled dot-product attention | `MultiHeadSelfAttentionScratch` |
| Multi-head split / merge projections | `MultiHeadSelfAttentionScratch` |
| Sinusoidal positional encoding | `sinusoidal_positional_encoding` |
| Layer normalisation | `LayerNormScratch` |
| Position-wise feed-forward + residual block | `EncoderBlockScratch` |

**Not used anywhere in the models:** `nn.LSTM`, `nn.GRU`, `nn.RNN`, `nn.MultiheadAttention`,
`nn.Transformer*`, `nn.LayerNorm`, `F.scaled_dot_product_attention`. `nn.LSTM` and
`F.layer_norm` appear **once each, inside Section 5**, used as a numerical reference to prove the
hand-written layers are correct.

**Structure.** 1 Data → 2 Controlled experiment setup → 3 BiLSTM from scratch →
4 Transformer from scratch → 5 Correctness tests → 6 Shared training harness →
7 Comparative evaluation → 8 Conclusion.

## 0. Configuration and reproducibility

`SMOKE_TEST = True` shrinks the whole notebook to a ~1 minute end-to-end run (2,000 records,
1 epoch) — useful for checking the notebook executes before committing to the full run.
Set it to `False` for the real experiment.

In [ ]:
SMOKE_TEST = False   # True -> 2,000 records / 1 epoch, for a fast end-to-end check

# ---- experiment configuration -------------------------------------------------
SEED          = 42
SUBSET_SIZE   = 60_000     # controlled subset drawn from the 120k training file
VAL_FRACTION  = 0.10       # -> 54,000 train / 6,000 validation
MAX_LEN       = 64         # tokens per document (truncate / pad)
MIN_FREQ      = 2          # vocabulary cutoff, computed on the training split only

# ---- shared training budget (identical for both models) -----------------------
BATCH_SIZE    = 128
MAX_EPOCHS    = 8
PATIENCE      = 2          # early stopping on validation macro-F1
GRAD_CLIP     = 1.0
WEIGHT_DECAY  = 0.01

# ---- per-architecture learning rates (tuned on validation only) ---------------
LR_BILSTM      = 1e-3
LR_TRANSFORMER = 3e-4

# ---- architecture sizes -------------------------------------------------------
EMB_DIM        = 128       # shared embedding width, so both models see the same input capacity
LSTM_HIDDEN    = 128
LSTM_DROPOUT   = 0.30
D_MODEL        = 128
N_HEADS        = 4
N_LAYERS       = 2
D_FF           = 256
TRF_DROPOUT    = 0.10

if SMOKE_TEST:
    SUBSET_SIZE, MAX_EPOCHS, PATIENCE = 2_000, 1, 1

LABELS = ["World", "Sports", "Business", "Sci/Tech"]
N_CLASSES = len(LABELS)

DATA_DIR = "data/agnews"
ART_DIR  = "artifacts"

print(f"Mode: {'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}  |  subset={SUBSET_SIZE:,}  epochs={MAX_EPOCHS}")

In [ ]:
import json, math, os, random, re, time, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import binomtest
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
Path(ART_DIR).mkdir(exist_ok=True)


def set_seed(seed: int = SEED) -> None:
    # Seed every source of randomness we touch, so the whole notebook is reproducible.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device: {DEVICE} | threads: {torch.get_num_threads()}")

In [ ]:
# ---- plotting style ------------------------------------------------------------
# Categorical palette in fixed slot order (blue, orange, aqua, yellow). Colours are
# assigned by entity (model / class) and never re-assigned when a chart's series change.
SERIES   = {"BiLSTM": "#2a78d6", "Transformer": "#eb6834"}
CLASS_C  = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
INK      = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#8a8880"}
GRID     = "#e3e2de"
SURFACE  = "#fcfcfb"
# Single-hue sequential ramp (light -> dark blue) for magnitude, e.g. confusion matrices.
BLUE_SEQ = LinearSegmentedColormap.from_list(
    "blue_seq", ["#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.linewidth": 0.8, "axes.labelcolor": INK["secondary"],
    "axes.titlecolor": INK["primary"], "axes.titlesize": 12, "axes.titleweight": "600",
    "axes.titlelocation": "left", "axes.titlepad": 12, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK["secondary"], "ytick.color": INK["secondary"],
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "xtick.bottom": True, "ytick.left": True, "xtick.direction": "out",
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.alpha": 1.0,
    "legend.frameon": False, "legend.fontsize": 9, "legend.labelcolor": INK["secondary"],
    "lines.linewidth": 2.0, "lines.markersize": 8,
    "font.size": 10, "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
})


def recede(ax, axis="y"):
    # Grid and axes stay recessive: one faint set of gridlines behind the data, nothing else.
    ax.grid(axis=axis, zorder=0)
    ax.set_axisbelow(True)
    return ax


def save_fig(fig, name):
    path = Path(ART_DIR) / name
    fig.savefig(path)
    print(f"saved -> {path}")

## 1. Data acquisition

The dataset is pulled through the Kaggle API so the source is reproducible and citable. The
newer Kaggle token format (`KGAT_...`) is read from `~/.kaggle/access_token` or the
`KAGGLE_API_TOKEN` environment variable.

The CSVs are downloaded into their own directory and only ever read with `pandas.read_csv` —
nothing from the download directory is executed.

In [ ]:
def ensure_agnews(data_dir: str = DATA_DIR) -> tuple[Path, Path]:
    # Download AG News from Kaggle unless it is already on disk.
    train_csv, test_csv = Path(data_dir) / "train.csv", Path(data_dir) / "test.csv"
    if train_csv.exists() and test_csv.exists():
        print(f"Found cached dataset in {data_dir}/ - skipping download.")
        return train_csv, test_csv

    token_file = Path.home() / ".kaggle" / "access_token"
    legacy     = Path.home() / ".kaggle" / "kaggle.json"
    if not (token_file.exists() or legacy.exists() or os.environ.get("KAGGLE_API_TOKEN")):
        raise RuntimeError(
            "No Kaggle credentials found. Do one of:\n"
            "  mkdir -p ~/.kaggle && echo KGAT_xxx > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token\n"
            "  export KAGGLE_API_TOKEN=KGAT_xxx\n"
            "  (legacy) place username/key JSON at ~/.kaggle/kaggle.json with chmod 600\n"
            "Get a token from https://www.kaggle.com/settings")

    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    Path(data_dir).mkdir(parents=True, exist_ok=True)
    api.dataset_download_files("amananandrai/ag-news-classification-dataset",
                               path=data_dir, unzip=True, quiet=True)
    print(f"Downloaded AG News to {data_dir}/")
    return train_csv, test_csv


TRAIN_CSV, TEST_CSV = ensure_agnews()

In [ ]:
# The raw CSVs carry literal escape artefacts from the original scrape, e.g.
# "dwindling\band of ultra-cynics" and "\n" inside Description. Left in place, the
# tokeniser would weld two real words into one junk type, so they are stripped first.
_ESCAPES = re.compile(r"\\[a-z]")


def load_split(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"Class Index": "class_index"})
    title = df["Title"].fillna("").astype(str)
    desc  = df["Description"].fillna("").astype(str)
    text  = (title.str.strip() + ". " + desc.str.strip())
    text  = text.str.replace(_ESCAPES, " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    out = pd.DataFrame({"text": text, "label": df["class_index"].astype(int) - 1})
    before = len(out)
    out = out[(out["text"].str.len() > 0) & out["label"].between(0, N_CLASSES - 1)].reset_index(drop=True)
    print(f"{csv_path.name}: {before:,} rows -> {len(out):,} usable ({before - len(out)} dropped)")
    return out


train_full = load_split(TRAIN_CSV)
test_df    = load_split(TEST_CSV)

print("\nExample records:")
for i in [0, 1, 2]:
    print(f"  [{LABELS[train_full.label[i]]:<9}] {train_full.text[i][:95]}...")

## 2. The controlled experiment

Three things make this a *controlled* comparison rather than two unrelated training runs:

1. **One stratified 60,000-record subset** (15,000 per class) drawn from the training file with a
   fixed seed, split 54,000 / 6,000 into train and validation.
2. **One shared preprocessing pipeline** — the same tokeniser, the same vocabulary (built on the
   training split only, so no leakage), the same `MAX_LEN`, and the same integer tensors are fed to
   both models.
3. **One shared training harness** (Section 6) with an identical budget: same batch size, same
   maximum epochs, same optimiser family, same early-stopping rule, same seed.

The only thing that differs between the two runs is **the architecture** (and its learning rate,
discussed in Section 6).

In [ ]:
set_seed()

# Stratified subset: exactly SUBSET_SIZE // 4 records per class.
per_class = SUBSET_SIZE // N_CLASSES
class_samples = [train_full[train_full.label == c].sample(
                     n=min(per_class, int((train_full.label == c).sum())), random_state=SEED)
                 for c in range(N_CLASSES)]
subset = (pd.concat(class_samples)
            .sample(frac=1.0, random_state=SEED)      # shuffle so splits are not class-ordered
            .reset_index(drop=True))

train_df, val_df = train_test_split(
    subset, test_size=VAL_FRACTION, random_state=SEED, stratify=subset["label"])
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

dist = pd.DataFrame({
    "Train": train_df.label.value_counts().sort_index(),
    "Validation": val_df.label.value_counts().sort_index(),
    "Test (official)": test_df.label.value_counts().sort_index(),
}).rename(index=dict(enumerate(LABELS)))
dist.loc["TOTAL"] = dist.sum()
print("Class distribution across splits\n")
print(dist.to_string())
print(f"\nControlled subset: {len(subset):,} records -> {len(train_df):,} train / {len(val_df):,} val")
print(f"Held-out official test set: {len(test_df):,} records")

In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9']+")
PAD_ID, UNK_ID = 0, 1


def tokenize(text: str) -> list[str]:
    # Lowercase + keep alphanumeric/apostrophe runs. Deliberately simple: both models get the
    # same tokens, so tokeniser sophistication cannot favour one architecture over the other.
    return TOKEN_RE.findall(text.lower())


# Vocabulary from the TRAINING split only.
counts = Counter()
train_tokens = [tokenize(t) for t in train_df.text]
for toks in train_tokens:
    counts.update(toks)

itos = ["<pad>", "<unk>"] + [w for w, c in counts.most_common() if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
VOCAB_SIZE = len(itos)

print(f"Token types seen in training split : {len(counts):,}")
print(f"Vocabulary (min_freq >= {MIN_FREQ})       : {VOCAB_SIZE:,}")
print(f"Most common: {[w for w, _ in counts.most_common(12)]}")

In [ ]:
lengths = np.array([len(t) for t in train_tokens])
coverage = (lengths <= MAX_LEN).mean()

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.hist(lengths, bins=np.arange(0, 121, 2), color=SERIES["BiLSTM"], edgecolor=SURFACE, linewidth=0.6, zorder=2)
ax.axvline(MAX_LEN, color=INK["primary"], linestyle="--", linewidth=1.5, zorder=3)
ax.annotate(f"MAX_LEN = {MAX_LEN}\n{coverage:.1%} of documents fit untruncated",
            xy=(MAX_LEN, ax.get_ylim()[1] * 0.92), xytext=(MAX_LEN + 5, ax.get_ylim()[1] * 0.92),
            color=INK["primary"], fontsize=9, va="top")
ax.set(title="Document length in tokens (training split)", xlabel="tokens", ylabel="documents", xlim=(0, 120))
recede(ax)
fig.tight_layout(); save_fig(fig, "fig0_token_lengths.png"); plt.show()

print(f"median={np.median(lengths):.0f}  mean={lengths.mean():.1f}  p95={np.percentile(lengths, 95):.0f}  max={lengths.max()}")
print(f"Coverage at MAX_LEN={MAX_LEN}: {coverage:.2%} of training documents are not truncated.")

In [ ]:
def encode(texts, max_len: int = MAX_LEN):
    # -> (ids[N, max_len] int64, lengths[N] int64). Right-padded with PAD_ID.
    n = len(texts)
    ids  = np.zeros((n, max_len), dtype=np.int64)
    lens = np.zeros(n, dtype=np.int64)
    for i, text in enumerate(texts):
        toks = tokenize(text)[:max_len]
        if not toks:                       # keep every row non-empty so masks never sum to 0
            toks = ["<unk>"]
        ids[i, :len(toks)] = [stoi.get(t, UNK_ID) for t in toks]
        lens[i] = len(toks)
    return torch.from_numpy(ids), torch.from_numpy(lens)


def make_dataset(df):
    ids, lens = encode(df.text.tolist())
    return TensorDataset(ids, lens, torch.tensor(df.label.values, dtype=torch.long))


t0 = time.time()
train_ds, val_ds, test_ds = make_dataset(train_df), make_dataset(val_df), make_dataset(test_df)
print(f"Encoded {len(train_ds) + len(val_ds) + len(test_ds):,} documents in {time.time() - t0:.1f}s")

g = torch.Generator().manual_seed(SEED)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g, drop_last=False)
val_dl   = DataLoader(val_ds,   batch_size=512, shuffle=False)
test_dl  = DataLoader(test_ds,  batch_size=512, shuffle=False)

xb, lb, yb = next(iter(train_dl))
print(f"batch ids {tuple(xb.shape)} | lengths {tuple(lb.shape)} | labels {tuple(yb.shape)}")
print(f"first row decoded: {' '.join(itos[i] for i in xb[0][:lb[0]])[:110]}...")

## 3. Model A — Bidirectional LSTM, written from scratch

The LSTM is built up in three layers of abstraction, none of which uses `nn.LSTM`:

**`LSTMCellScratch`** — one timestep. The four gates are computed in a single fused matmul and
split with `chunk(4)`:

$$\begin{aligned}
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) &\quad\text{(input gate)}\\
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) &\quad\text{(forget gate)}\\
g_t &= \tanh(W_g x_t + U_g h_{t-1} + b_g) &\quad\text{(candidate cell)}\\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) &\quad\text{(output gate)}\\
c_t &= f_t \odot c_{t-1} + i_t \odot g_t, \qquad h_t = o_t \odot \tanh(c_t)
\end{aligned}$$

The gate order `i, f, g, o` deliberately matches PyTorch's own convention so that Section 5 can
copy `nn.LSTM`'s weights straight across and compare outputs numerically.

**`LSTMLayerScratch`** — the explicit Python loop over timesteps, in either direction. Two details
matter and are the usual source of silent accuracy loss:

1. *Padding must not corrupt the state.* Each step gates the update by the mask,
   `h ← m·h_new + (1−m)·h_prev`. Without this, pad tokens keep driving the recurrence and the
   final state drifts away from the real end of the sentence.
2. *The backward pass must start at the true end of each sequence.* Because padding sits at the
   right and masked steps are exact no-ops, iterating `t = T−1 … 0` with the same masking is
   equivalent to starting at each sequence's own last real token — so the returned final state is
   the correct per-sequence endpoint for both directions.

**`BiLSTMClassifier`** — embedding → forward + backward layers → pooling → linear head. Pooling
concatenates the masked mean over all timesteps with the final hidden state of each direction
(`2H + H + H = 4H` features): mean-pooling alone discards the endpoint the recurrence worked to
build, while the endpoint alone is noisy on documents this short.

In [ ]:
class LSTMCellScratch(nn.Module):
    # One LSTM timestep. W: input->gates (with bias), U: hidden->gates.
    def __init__(self, d_in: int, d_hidden: int):
        super().__init__()
        self.d_hidden = d_hidden
        self.W = nn.Linear(d_in, 4 * d_hidden, bias=True)
        self.U = nn.Linear(d_hidden, 4 * d_hidden, bias=False)
        self.reset_parameters()

    def reset_parameters(self):
        # Classic LSTM init: U(-1/sqrt(H), 1/sqrt(H)) on every parameter.
        k = 1.0 / math.sqrt(self.d_hidden)
        for p in self.parameters():
            nn.init.uniform_(p, -k, k)

    def forward(self, x_t, h_prev, c_prev):
        gates = self.W(x_t) + self.U(h_prev)             # (B, 4H) in one matmul
        i, f, g, o = gates.chunk(4, dim=-1)              # PyTorch gate order: i, f, g, o
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_t = f * c_prev + i * g
        h_t = o * torch.tanh(c_t)
        return h_t, c_t


class LSTMLayerScratch(nn.Module):
    # Unrolls LSTMCellScratch over time in one direction, mask-aware.
    def __init__(self, d_in: int, d_hidden: int, reverse: bool = False):
        super().__init__()
        self.cell, self.d_hidden, self.reverse = LSTMCellScratch(d_in, d_hidden), d_hidden, reverse

    def forward(self, x, mask):
        # x (B, T, D); mask (B, T) float with 1.0 on real tokens.
        B, T, _ = x.shape
        h = x.new_zeros(B, self.d_hidden)
        c = x.new_zeros(B, self.d_hidden)
        steps = range(T - 1, -1, -1) if self.reverse else range(T)
        outputs = []
        for t in steps:
            m = mask[:, t].unsqueeze(1)                  # (B, 1)
            h_new, c_new = self.cell(x[:, t], h, c)
            # Gate the state update: a padded step leaves (h, c) exactly unchanged.
            h = m * h_new + (1.0 - m) * h
            c = m * c_new + (1.0 - m) * c
            outputs.append(h)
        if self.reverse:
            outputs.reverse()                            # back into position order
        return torch.stack(outputs, dim=1), h            # (B, T, H), (B, H) final valid state


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = EMB_DIM, hidden: int = LSTM_HIDDEN,
                 n_classes: int = N_CLASSES, dropout: float = LSTM_DROPOUT):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.1)
        with torch.no_grad():
            self.embedding.weight[PAD_ID].zero_()
        self.forward_layer  = LSTMLayerScratch(emb_dim, hidden, reverse=False)
        self.backward_layer = LSTMLayerScratch(emb_dim, hidden, reverse=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(4 * hidden, n_classes)

    def forward(self, ids, lengths=None):
        mask = (ids != PAD_ID).float()                           # (B, T)
        x = self.embedding(ids)
        out_f, h_f = self.forward_layer(x, mask)
        out_b, h_b = self.backward_layer(x, mask)
        seq = torch.cat([out_f, out_b], dim=-1)                  # (B, T, 2H)
        denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        mean_pooled = (seq * mask.unsqueeze(-1)).sum(dim=1) / denom
        features = torch.cat([mean_pooled, h_f, h_b], dim=-1)    # (B, 4H)
        return self.classifier(self.dropout(features))


print(BiLSTMClassifier(VOCAB_SIZE))

## 4. Model B — Transformer encoder, written from scratch

A pre-norm Transformer encoder, assembled from four hand-written pieces:

**`sinusoidal_positional_encoding`** — the original Vaswani et al. formulation, since
self-attention is permutation-invariant and would otherwise see a bag of words:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad
  PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

**`LayerNormScratch`** — normalises over the feature dimension with learned $\gamma, \beta$,
using the biased variance so it matches `F.layer_norm` exactly (verified in Section 5).

**`MultiHeadSelfAttentionScratch`** — scaled dot-product attention, with the head split and merge
done by hand:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}} + M\right)V$$

The key-padding mask $M$ is applied as a large negative constant (`-1e9`) rather than `-inf`:
with `-inf`, a row that is entirely masked would softmax to `NaN` and poison the gradients.

**`EncoderBlockScratch`** — pre-norm residuals, `x + attn(norm(x))` then `x + ffn(norm(x))`. Pre-norm
is the deliberate choice here: it trains stably without a learning-rate warmup schedule, which
matters when the budget is a handful of CPU epochs.

Capacity is deliberately matched to the BiLSTM so the comparison is architectural rather than a
disguised parameter-count contest — Section 7 reports both totals.

In [ ]:
def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    # (max_len, d_model) fixed, non-learned position signal.
    position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)        # (T, 1)
    even_i   = torch.arange(0, d_model, 2, dtype=torch.float32)               # (d_model/2,)
    div_term = torch.exp(-math.log(10000.0) * even_i / d_model)
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)[:, : pe[:, 1::2].shape[1]]
    return pe


class LayerNormScratch(nn.Module):
    # Normalises over the last dimension. Biased variance, to match F.layer_norm.
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        mu  = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mu) / torch.sqrt(var + self.eps) + self.beta


class MultiHeadSelfAttentionScratch(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads, self.d_k = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, t):
        B, T, _ = t.shape
        return t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)   # (B, h, T, d_k)

    def forward(self, x, key_mask=None, return_attention: bool = False):
        B, T, _ = x.shape
        q, k, v = self._split_heads(self.W_q(x)), self._split_heads(self.W_k(x)), self._split_heads(self.W_v(x))

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)     # (B, h, T, T)
        if key_mask is not None:
            # Broadcast (B, T) -> (B, 1, 1, T): every query ignores padded keys.
            scores = scores.masked_fill(key_mask[:, None, None, :] == 0, -1e9)

        attention = torch.softmax(scores, dim=-1)
        context = self.dropout(attention) @ v                        # (B, h, T, d_k)
        context = context.transpose(1, 2).reshape(B, T, self.n_heads * self.d_k)
        out = self.W_o(context)
        return (out, attention) if return_attention else out


class EncoderBlockScratch(nn.Module):
    # Pre-norm block: x + attn(norm(x)), then x + ffn(norm(x)).
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.norm1 = LayerNormScratch(d_model)
        self.attention = MultiHeadSelfAttentionScratch(d_model, n_heads, dropout)
        self.norm2 = LayerNormScratch(d_model)
        self.ff_in  = nn.Linear(d_model, d_ff)
        self.ff_out = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_mask=None):
        x = x + self.dropout(self.attention(self.norm1(x), key_mask))
        h = self.norm2(x)
        h = self.ff_out(self.dropout(F.gelu(self.ff_in(h))))
        return x + self.dropout(h)


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = D_MODEL, n_heads: int = N_HEADS,
                 n_layers: int = N_LAYERS, d_ff: int = D_FF, max_len: int = MAX_LEN,
                 n_classes: int = N_CLASSES, dropout: float = TRF_DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.1)
        with torch.no_grad():
            self.embedding.weight[PAD_ID].zero_()
        self.register_buffer("pos_encoding", sinusoidal_positional_encoding(max_len, d_model))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            EncoderBlockScratch(d_model, n_heads, d_ff, dropout) for _ in range(n_layers))
        self.norm_final = LayerNormScratch(d_model)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, ids, lengths=None, return_attention: bool = False):
        mask = (ids != PAD_ID)                                        # (B, T) bool
        T = ids.size(1)
        # Scale embeddings by sqrt(d_model) so they are not swamped by the positional signal.
        x = self.embedding(ids) * math.sqrt(self.d_model) + self.pos_encoding[:T].unsqueeze(0)
        x = self.dropout(x)

        attn_maps = []
        for block in self.blocks:
            if return_attention:
                normed = block.norm1(x)
                attended, attn = block.attention(normed, mask, return_attention=True)
                attn_maps.append(attn)
                x = x + block.dropout(attended)
                h = block.ff_out(block.dropout(F.gelu(block.ff_in(block.norm2(x)))))
                x = x + block.dropout(h)
            else:
                x = block(x, mask)
        x = self.norm_final(x)

        m = mask.float().unsqueeze(-1)
        pooled = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)     # masked mean over tokens
        logits = self.classifier(self.dropout(pooled))
        return (logits, attn_maps) if return_attention else logits


def count_parameters(model) -> dict:
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    emb   = sum(p.numel() for n, p in model.named_parameters() if "embedding" in n)
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


print(TransformerClassifier(VOCAB_SIZE))

## 5. Correctness tests for the hand-written layers

A from-scratch layer that is subtly wrong still trains and still produces a plausible-looking
accuracy number — it just quietly produces a *worse* one, which would invalidate the whole
comparison. These checks are therefore part of the deliverable, not scaffolding.

| Check | What it rules out |
|---|---|
| `LSTMCellScratch` vs. `nn.LSTM` with copied weights | wrong gate order, wrong gate algebra, misplaced bias |
| `LayerNormScratch` vs. `F.layer_norm` | unbiased-vs-biased variance, wrong reduction axis |
| `MultiHeadSelfAttentionScratch` vs. a naive per-head loop | wrong head split/merge reshape, wrong scaling |
| attention rows sum to 1 | softmax over the wrong axis |
| **padding invariance** | the two masking bugs called out in Section 3 |
| overfitting 200 examples | a broken backward path / dead gradients |

> `nn.LSTM` and `F.layer_norm` appear **only in this section**, purely as numerical references.
> Neither is used by either model.

In [ ]:
set_seed(0)
results = {}

# ---- Check 1: LSTMCellScratch / LSTMLayerScratch vs. nn.LSTM ------------------
# nn.LSTM is used HERE ONLY as a reference implementation, never inside our models.
B, T, D, H = 4, 7, 10, 16
ref_lstm = nn.LSTM(D, H, num_layers=1, batch_first=True, bidirectional=False)
ours = LSTMLayerScratch(D, H, reverse=False)
with torch.no_grad():
    # nn.LSTM packs gates as [i, f, g, o] - the same order LSTMCellScratch chunks into.
    ours.cell.W.weight.copy_(ref_lstm.weight_ih_l0)
    ours.cell.W.bias.copy_(ref_lstm.bias_ih_l0 + ref_lstm.bias_hh_l0)   # our U has no bias
    ours.cell.U.weight.copy_(ref_lstm.weight_hh_l0)

x = torch.randn(B, T, D)
full_mask = torch.ones(B, T)
with torch.no_grad():
    ref_out, (ref_h, _) = ref_lstm(x)
    our_out, our_h = ours(x, full_mask)

d_out, d_h = (ref_out - our_out).abs().max().item(), (ref_h[0] - our_h).abs().max().item()
results["LSTM cell matches nn.LSTM"] = (max(d_out, d_h) < 1e-5, f"max|diff| outputs={d_out:.2e} final h={d_h:.2e}")

# ---- Check 2: LayerNormScratch vs. F.layer_norm --------------------------------
ln = LayerNormScratch(32)
with torch.no_grad():
    nn.init.normal_(ln.gamma, 1.0, 0.2)
    nn.init.normal_(ln.beta, 0.0, 0.2)
    z = torch.randn(5, 9, 32) * 3.0 + 1.5
    d_ln = (ln(z) - F.layer_norm(z, (32,), ln.gamma, ln.beta, eps=1e-5)).abs().max().item()
results["LayerNorm matches F.layer_norm"] = (d_ln < 1e-6, f"max|diff|={d_ln:.2e}")

In [ ]:
# ---- Check 3: multi-head attention vs. a naive per-head loop -------------------
# The reference recomputes attention one (batch, head) pair at a time with explicit
# column slicing, which independently verifies the view/transpose head split and merge.
set_seed(0)
d_model, n_heads, Tq = 24, 3, 6
mha = MultiHeadSelfAttentionScratch(d_model, n_heads, dropout=0.0).eval()
x = torch.randn(3, Tq, d_model)
km = torch.ones(3, Tq, dtype=torch.bool)
km[0, 4:] = False          # first row has two padded positions
km[2, 5:] = False

with torch.no_grad():
    out, attn = mha(x, km, return_attention=True)
    q_all, k_all, v_all = mha.W_q(x), mha.W_k(x), mha.W_v(x)
    d_k = d_model // n_heads
    naive = torch.zeros_like(out)
    for b in range(x.size(0)):
        head_outputs = []
        for h in range(n_heads):
            sl = slice(h * d_k, (h + 1) * d_k)
            qh, kh, vh = q_all[b, :, sl], k_all[b, :, sl], v_all[b, :, sl]
            s = (qh @ kh.T) / math.sqrt(d_k)
            s = s.masked_fill(~km[b].unsqueeze(0), -1e9)
            head_outputs.append(torch.softmax(s, dim=-1) @ vh)
        naive[b] = mha.W_o(torch.cat(head_outputs, dim=-1))

d_attn = (out - naive).abs().max().item()
results["Attention matches naive per-head reference"] = (d_attn < 1e-5, f"max|diff|={d_attn:.2e}")

row_sums = attn.sum(dim=-1)
results["Attention rows sum to 1"] = (
    torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5),
    f"max|rowsum-1|={(row_sums - 1).abs().max().item():.2e}")

# Padded keys must receive essentially zero attention mass from every query.
pad_mass = attn[0, :, :, 4:].max().item()
results["Padded keys get ~zero attention"] = (pad_mass < 1e-6, f"max mass on a padded key={pad_mass:.2e}")

In [ ]:
# ---- Check 4: padding invariance (the decisive masking test) -------------------
# Appending extra <pad> tokens must not move either model's logits by more than float noise.
set_seed(0)
bilstm_t = BiLSTMClassifier(VOCAB_SIZE).eval()
trf_t    = TransformerClassifier(VOCAB_SIZE).eval()

short_ids = torch.randint(2, 500, (6, 12))
short_ids[0, 8:] = PAD_ID          # mixed real lengths within the batch
short_ids[3, 5:] = PAD_ID
padded_ids = torch.cat([short_ids, torch.full((6, 20), PAD_ID, dtype=torch.long)], dim=1)

with torch.no_grad():
    for name, model in [("BiLSTM", bilstm_t), ("Transformer", trf_t)]:
        d = (model(short_ids) - model(padded_ids)).abs().max().item()
        results[f"{name}: logits unchanged by extra padding"] = (d < 1e-4, f"max|diff|={d:.2e}")

# ---- Check 5: both models can overfit 200 examples (backward path is alive) ----
sub_ids, sub_lens, sub_y = train_ds[:200]
for name, model, lr in [("BiLSTM", BiLSTMClassifier(VOCAB_SIZE), 3e-3),
                        ("Transformer", TransformerClassifier(VOCAB_SIZE), 3e-3)]:
    set_seed()
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(30):
        opt.zero_grad()
        loss = F.cross_entropy(model(sub_ids, sub_lens), sub_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
    model.eval()
    with torch.no_grad():
        acc = (model(sub_ids, sub_lens).argmax(-1) == sub_y).float().mean().item()
    results[f"{name}: overfits 200 examples"] = (acc > 0.95, f"train accuracy={acc:.1%} after 30 steps")

# ---- report -------------------------------------------------------------------
print(f"{'check':<48} {'result':<7} detail")
print("-" * 100)
failed = 0
for name, (ok, detail) in results.items():
    failed += not ok
    print(f"{name:<48} {'PASS' if ok else 'FAIL':<7} {detail}")
print("-" * 100)
assert failed == 0, f"{failed} correctness check(s) failed - fix before training."
print(f"All {len(results)} checks passed. The hand-written layers are numerically correct.")

del bilstm_t, trf_t, ref_lstm, ours, mha